# Amazon Bedrock Knowledge Bases and Strands Agents

This notebook demonstrates how to create an Amazon OpenSearch Serverless collection and an Amazon Bedrock Knowledge Base using the AWS SDK for Python (Boto3), then build an autonomous agent over them with the open-source Strands Agents SDK.

## Step 1: Install dependencies and initialize clients

In [1]:
# Install required packages: Installs boto3, the AWS SDK for Python, 
# which is needed to interact with AWS services programmatically through API calls.

!pip install boto3

Defaulting to user installation because normal site-packages is not writeable


In [2]:
# Import libraries: Imports necessary Python modules for AWS interactions, 
# timing operations for polling, and JSON handling for policy definitions.

import os
import boto3
import time
import json


<i aria-hidden="true" class="fas fa-exclamation-circle" style="color:#7C5AED"></i> **Caution:** Before running the next code cell, replace the following placeholder variables with their actual values that you copied earlier. Those values are provided to the left of the main lab instructions.

- BedrockKnowledgeBaseRoleArn
- S3BucketBKB

In [ ]:
# Initialize clients: Sets up the OpenSearchServerless client and defines 
# key variables including role ARNs and S3 bucket that will be used throughout the notebook. 
# Gets the current region and execution role to ensure proper AWS resource access and permissions.

client = boto3.client('opensearchserverless')

# Update with outputs for lab in CloudFormation
bedrock_bkb_role_arn = ''
s3_bucket = ''

# Get current region 
curr_session = boto3.session.Session()
curr_region = curr_session.region_name

# Get current execution role: derive the IAM role ARN of this Code Editor
# instance from the STS caller identity (arn:aws:sts::<acct>:assumed-role/<role>/<session>)
caller_arn = boto3.client('sts').get_caller_identity()['Arn']
account_id, role_name = caller_arn.split(':')[4], caller_arn.split('/')[-2]
exec_role = f"arn:aws:iam::{account_id}:role/{role_name}"


## Step 2: Create a network policy for the collection

In [4]:
# Define collection name: Names the OpenSearch Serverless collection 
# that will be created and used throughout the lab.

collection_name = 'skillbuilder'

# Create network policy: Creates a security policy that allows public access 
# to the collection and dashboard, which is necessary for Bedrock to access 
# the vector store. This enables the Knowledge Base to query the collection.

network_policy_json = json.dumps([
    {
        "Rules": [
            {
                "ResourceType": "dashboard",
                "Resource": [f"collection/{collection_name}"],
            },
            {
                "ResourceType": "collection",
                "Resource": [f"collection/{collection_name}"],
            }
        ],
        "AllowFromPublic": True
    }
])

try:
    response = client.create_security_policy(
        name=f"{collection_name}-network-policy",
        type='network',
        description='Network policy for OpenSearch Serverless collection',
        policy=network_policy_json
    )
    print(f"Network policy created: {response['securityPolicyDetail']['name']}")
except client.exceptions.ConflictException:
    print(f"Network policy '{collection_name}-network-policy' already exists, skipping.")

Network policy created: skillbuilder-network-policy


## Step 3: Create a data access policy for the collection

In [5]:
# Create data access policy: Defines permissions for who can access the collection data. 
# Grants full permissions to both the Bedrock Knowledge Base role and the notebook's 
# execution role to perform operations on the collection and its indexes. 
# This ensures proper authorization for data operations.

data_policy_json = json.dumps([
    {
        "Rules": [
            {
                "ResourceType": "collection",
                "Resource": [f"collection/{collection_name}"],
                "Permission": [
                    "aoss:*"
                ]
            },
            {
                "ResourceType": "index",
                "Resource": [f"index/{collection_name}/*"],
                "Permission": [
                    "aoss:*"
                ]
            }
        ],
        "Principal": [bedrock_bkb_role_arn, exec_role]
    }
])

try:
    response = client.create_access_policy(
        name=f"{collection_name}-data-policy",
        type='data',
        description='Data policy for OpenSearch Serverless collection',
        policy=data_policy_json
    )
    print(f"Data policy created: {response['accessPolicyDetail']['name']}")
except client.exceptions.ConflictException:
    print(f"Data policy '{collection_name}-data-policy' already exists, skipping.")

Data policy created: skillbuilder-data-policy


## Step 4: Create an encryption policy

In [6]:
# Create an encryption policy: Sets up encryption for the OpenSearch collection 
# using AWS-owned keys, ensuring data is encrypted at rest as a security best practice. 
# This protects the vector data stored in the collection.

encryption_policy_json = json.dumps(
    {
        "Rules": [
            {
                "ResourceType": "collection",
                "Resource": [f"collection/{collection_name}"]
            }
        ],
        "AWSOwnedKey": True
    }
)

try:
    response = client.create_security_policy(
        name=f"{collection_name}-encryption-policy",
        type="encryption",
        description="Encryption policy for OpenSearch Serverless collection",
        policy=encryption_policy_json
    )
    print(f"Encryption policy created: {response['securityPolicyDetail']['name']}")
except client.exceptions.ConflictException:
    print(f"Encryption policy '{collection_name}-encryption-policy' already exists, skipping.")

Encryption policy created: skillbuilder-encryption-policy


## Step 5: Create the OpenSearch Serverless collection

In [ ]:
# Create the collection: Initializes a VECTORSEARCH type OpenSearch Serverless 
# collection that will store vector embeddings for the Knowledge Base. 
# This specialized collection type is optimized for vector similarity searches.

try:
    response = client.create_collection(
        name=collection_name,
        type='VECTORSEARCH',
        description='OpenSearch Serverless collection for a Bedrock Knowledge Base'
    )
    collection_id = response['createCollectionDetail']['id']
    print(f"Collection creation initiated. Collection ID: {collection_id}")
except client.exceptions.ConflictException:
    response = client.batch_get_collection(names=[collection_name])
    collection_id = response['collectionDetails'][0]['id']
    print(f"Collection '{collection_name}' already exists. Collection ID: {collection_id}")

In [8]:
# Wait for collection: Implements a polling mechanism to check the collection's 
# status until it becomes ACTIVE, as collection creation is an asynchronous operation 
# that can take several minutes to complete.

print("Waiting for collection to become active...")
status = None
while True:
    response = client.batch_get_collection(ids=[collection_id])
    status = response['collectionDetails'][0]['status']
    print(f"Current status: {status}")
    
    if status == 'ACTIVE':
        break
    elif status in ['FAILED', 'DELETED']:
        print(f"Collection creation failed with status: {status}")
        break
        
    time.sleep(15)

Waiting for collection to become active...
Current status: CREATING
Current status: CREATING
Current status: CREATING
Current status: CREATING
Current status: CREATING
Current status: CREATING
Current status: CREATING
Current status: CREATING
Current status: CREATING
Current status: CREATING
Current status: CREATING
Current status: CREATING
Current status: CREATING
Current status: CREATING
Current status: ACTIVE


## Step 6: Get collection endpoint

In [ ]:
# Get collection endpoint: Retrieves and displays the endpoints for the 
# collection and dashboard once the collection is active, which will be needed 
# to configure the Knowledge Base connection to OpenSearch Serverless.

if status == 'ACTIVE':
    response = client.batch_get_collection(ids=[collection_id])
    dashboard_endpoint = response['collectionDetails'][0]['dashboardEndpoint']
    collection_endpoint = response['collectionDetails'][0]['collectionEndpoint']
    
    print(f"\nCollection successfully created!")
    print(f"Collection Name: {collection_name}")
    print(f"Collection ID: {collection_id}")
    print(f"Dashboard Endpoint: {dashboard_endpoint}")
    print(f"Collection Endpoint: {collection_endpoint}")
    print(f"Bedrock Knowledge Base Role ARN: {bedrock_bkb_role_arn}")

## Step 7: Create the vector index

In [10]:
# Install required packages for OpenSearch : Installs the Python requests 
# and requests-aws4auth libraries needed to make authenticated API calls 
# to OpenSearch Serverless.

!pip install requests requests-aws4auth

Defaulting to user installation because normal site-packages is not writeable


In [11]:
import requests
from requests_aws4auth import AWS4Auth

# Create AWS authentication for OpenSearch : Sets up AWS Signature Version 4 
# authentication (AWS4Auth) which is required to securely access the OpenSearch APIs. 
# This uses your current AWS credentials to sign requests to the OpenSearch Serverless endpoints.

credentials = boto3.Session().get_credentials()
region = boto3.Session().region_name
awsauth = AWS4Auth(
    credentials.access_key,
    credentials.secret_key,
    region,
    'aoss',
    session_token=credentials.token
)

### Define the index mapping with vector field for Amazon Titan V2 Embeddings

In [12]:
# Define the index mapping with vector field: Creates a specialized index configuration 
# optimized for vector embeddings from Amazon Titan V2. The mapping includes:

# A knn_vector field with 1024 dimensions (matching Titan V2's embedding size)

# FAISS as the vector search engine with HNSW algorithm for efficient similarity search

# Special fields for storing document text chunks and metadata that Bedrock Knowledge Base will use

index_mapping = {
  "settings": {
    "index": {
      "knn": "true"
    }
  },
  "mappings": {
    "dynamic_templates": [
      {
        "strings": {
          "match_mapping_type": "string",
          "mapping": {
            "fields": {
              "keyword": {
                "ignore_above": 2147483647,
                "type": "keyword"
              }
            },
            "type": "text"
          }
        }
      }
    ],
    "properties": {
      "bedrock-knowledge-base-default-vector": {
        "type": "knn_vector",
        "dimension": 1024,
        "method": {
          "engine": "faiss",
          "space_type": "l2",
          "name": "hnsw",
          "parameters": {}
        }
      },
      "AMAZON_BEDROCK_METADATA": {
        "type": "text",
        "index": "false"
      },
      "AMAZON_BEDROCK_TEXT_CHUNK": {
        "type": "text",
        "index": "true"
      }
    }
  }
}

In [ ]:
# Create the index
index_name = f"{collection_name}-index"

# Print the endpoint to debug
print(f"Collection endpoint: {collection_endpoint}")

# Fix URL construction
if collection_endpoint.startswith('https://'):
    url = f"{collection_endpoint}/{index_name}"
else:
    url = f"https://{collection_endpoint}/{index_name}"

print(f"Using URL: {url}")

# Create the index with a Put request
response = requests.put(url, auth=awsauth, json=index_mapping, headers={"Content-Type": "application/json"})
print(f"Index creation response: {response.status_code}")
print(response.text)

## Step 8: Create a Bedrock Knowledge Base

In [14]:
# Initialize the Bedrock Agent client - Creates a boto3 client used to build and 
# manage the Bedrock Knowledge Base (create knowledge base, data source, and ingestion).

bedrock_client = boto3.client('bedrock-agent')

In [ ]:
# Create a Bedrock Knowledge Base: Sets up a Knowledge Base that uses the OpenSearch Serverless 
# collection as its vector store and configures it to use Amazon Titan Embeddings V2 model 
# for converting text to vectors.

response = bedrock_client.create_knowledge_base(
    name=f"{collection_name}-kb",
    description="Knowledge base for SkillBuilder lab",
    roleArn=bedrock_bkb_role_arn,
    knowledgeBaseConfiguration={
        "type": "VECTOR",
        "vectorKnowledgeBaseConfiguration": {
            "embeddingModelArn": f"arn:aws:bedrock:{curr_region}::foundation-model/amazon.titan-embed-text-v2:0"
        }
    },
    storageConfiguration={
        "type": "OPENSEARCH_SERVERLESS",
        "opensearchServerlessConfiguration": {
            "collectionArn": f"arn:aws:aoss:{curr_region}:{boto3.client('sts').get_caller_identity()['Account']}:collection/{collection_id}",
            "vectorIndexName": f"{collection_name}-index",
            "fieldMapping": {
                "vectorField": "bedrock-knowledge-base-default-vector",
                "textField": "AMAZON_BEDROCK_TEXT_CHUNK",
                "metadataField": "AMAZON_BEDROCK_METADATA"
            }
        }
    }
)

knowledge_base_id = response['knowledgeBase']['knowledgeBaseId']
print(f"Knowledge Base creation initiated. Knowledge Base ID: {knowledge_base_id}")

<i aria-hidden="true" class="fas fa-exclamation-circle" style="color:#7C5AED"></i> **Caution:** If you see the following error, please re-run the code cell:

*ValidationException: An error occurred (ValidationException) when calling the CreateKnowledgeBase operation: The 
knowledge base storage configuration provided is invalid... Dependency error document status code: 404, error 
message: no such index [skillbuilder-index]*

In [16]:
# Wait for Knowledge Base activation: Implements a polling mechanism to check 
# the Knowledge Base status until it becomes active, as creation is an asynchronous operation.

print("Waiting for knowledge base to become active...")
kb_status = None
while True:
    response = bedrock_client.get_knowledge_base(knowledgeBaseId=knowledge_base_id)
    kb_status = response['knowledgeBase']['status']
    print(f"Current status: {kb_status}")
    
    if kb_status == 'ACTIVE':
        break
    elif kb_status in ['FAILED', 'DELETED']:
        print(f"Knowledge Base creation failed with status: {kb_status}")
        break
        
    time.sleep(30)

Waiting for knowledge base to become active...
Current status: ACTIVE


## Step 9: Create a Data Source and Start Ingestion

In [17]:
# Upload PDF file to S3: Uploads the Employee IT handbook PDF to the S3 bucket that will serve as 
# the data source for the Knowledge Base.

s3_client = boto3.client('s3')
pdf_file_path = 'Fictitious-Company-Employee-IT-Handbook.pdf'
s3_key = 'Fictitious-Company-Employee-IT-Handbook.pdf'

print(f"Uploading {pdf_file_path} to s3://{s3_bucket}/{s3_key}")
s3_client.upload_file(pdf_file_path, s3_bucket, s3_key)
print("Upload complete!")

Uploading Fictitious-Company-Employee-IT-Handbook.pdf to s3://labstack-8295a149-5b04-4010-a01d-194f5-s3bucketbkb-e77okadca08h/Fictitious-Company-Employee-IT-Handbook.pdf
Upload complete!


In [ ]:
# Create a data source: Configures a data source that connects the Knowledge Base 
# to the S3 bucket containing the documents.

response = bedrock_client.create_data_source(
    knowledgeBaseId=knowledge_base_id,
    name=f"{collection_name}-data-source",
    description="S3 data source for the knowledge base",
    dataSourceConfiguration={
        "type": "S3",
        "s3Configuration": {
            "bucketArn": f"arn:aws:s3:::{s3_bucket}"
        }
    },
    vectorIngestionConfiguration={
        "chunkingConfiguration": {
            "chunkingStrategy": "FIXED_SIZE",
            "fixedSizeChunkingConfiguration": {
                "maxTokens": 300,
                "overlapPercentage": 20
            }
        }
    }
)

data_source_id = response['dataSource']['dataSourceId']
print(f"Data source created. Data Source ID: {data_source_id}")

In [19]:
# Wait for the data source to become active
print("Waiting for data source to become active...")
ds_status = None
while True:
    response = bedrock_client.get_data_source(knowledgeBaseId=knowledge_base_id, dataSourceId=data_source_id)
    ds_status = response['dataSource']['status']
    print(f"Current status: {ds_status}")
    
    if ds_status == 'AVAILABLE':
        print(f"Current status: {ds_status}")
        break
    elif ds_status in ['DELETING', 'DELETE_UNSUCCESSFUL']:
        print(f"Data source creation failed with status: {ds_status}")
        break
        
    time.sleep(30)

Waiting for data source to become active...
Current status: AVAILABLE
Current status: AVAILABLE


In [ ]:
# Start ingestion job: Initiates the process of extracting text from documents, chunking it, 
# creating embeddings, and storing them in the vector index.

response = bedrock_client.start_ingestion_job(
    knowledgeBaseId=knowledge_base_id,
    dataSourceId=data_source_id
)

ingestion_job_id = response['ingestionJob']['ingestionJobId']
print(f"Ingestion job started. Ingestion Job ID: {ingestion_job_id}")

In [22]:
# Monitor ingestion job: Tracks the progress of document processing, showing statistics 
# on successful and failed documents.

print("Monitoring ingestion job status...")
job_status = None
while True:
    response = bedrock_client.get_ingestion_job(
        knowledgeBaseId=knowledge_base_id,
        dataSourceId=data_source_id,
        ingestionJobId=ingestion_job_id
    )
    job_status = response['ingestionJob']['status']
    statistics = response['ingestionJob'].get('statistics', {})
    
    print(f"Current status: {job_status}")
    if statistics:
        print(f"Documents processed: {statistics.get('numberOfDocumentsProcessed', 0)}")
        print(f"Documents failed: {statistics.get('numberOfDocumentsFailed', 0)}")
    
    if job_status in ['COMPLETE', 'FAILED', 'STOPPED']:
        break
        
    time.sleep(15)

Monitoring ingestion job status...
Current status: COMPLETE
Documents processed: 0
Documents failed: 0


## Step 10: Build an agent with the Strands Agents SDK

You now build an autonomous agent using the open-source [Strands Agents SDK](https://strandsagents.com/). A Strands agent is defined in code from three things: a **model**, a set of **tools**, and a **system prompt**. The agent runs a reasoning loop that decides when to search the Knowledge Base and when to call your tools.

For this lab the agent uses:
- The **Amazon Nova Lite** foundation model in Amazon Bedrock.
- The built-in `retrieve` tool, which performs semantic search against the Bedrock Knowledge Base you created.
- Two custom `@tool` functions, `create_ticket` and `lookup_ticket`, which call the ITSM Lambda functions that were provisioned for this lab.

In [23]:
# Install the Strands Agents SDK and its example tools package.
# strands-agents provides the Agent and BedrockModel classes; strands-agents-tools
# provides the prebuilt 'retrieve' tool for Amazon Bedrock Knowledge Bases.

!pip install strands-agents strands-agents-tools

Defaulting to user installation because normal site-packages is not writeable


In [24]:
# Build the agent: Defines the model, the tools, and the system prompt, then
# assembles them into a Strands Agent. No managed agent resource is created —
# the agent runs entirely in this notebook.

from strands import Agent, tool
from strands.models import BedrockModel
from strands_tools import retrieve

# The retrieve tool reads the target Knowledge Base and Region from the environment.
os.environ['KNOWLEDGE_BASE_ID'] = knowledge_base_id
os.environ['AWS_REGION'] = curr_region

# Client used by the custom tools to invoke the ITSM Lambda functions.
lambda_client = boto3.client('lambda')


@tool
def create_ticket(tickettype: str, description: str, impact: str, urgency: str) -> str:
    """Create an ITSM ticket.

    Args:
        tickettype: The type of ticket to create, for example INC or CHG.
        description: A description of the issue or request.
        impact: The impact of the issue or request.
        urgency: The urgency of the request.
    Returns:
        The created ticket number.
    """
    result = lambda_client.invoke(
        FunctionName='SkillBuilder-ITSM-Create',
        Payload=json.dumps({
            'tickettype': tickettype,
            'description': description,
            'impact': impact,
            'urgency': urgency
        }).encode('utf-8')
    )
    payload = json.loads(result['Payload'].read())
    return payload.get('ticketNumber', str(payload))


@tool
def lookup_ticket(ticketNumber: str) -> dict:
    """Look up the status and details of an ITSM ticket by its ticket number.

    Args:
        ticketNumber: The ticket number provided by the user.
    Returns:
        A dictionary with the ticket status, description, impact, urgency, and created date.
    """
    result = lambda_client.invoke(
        FunctionName='SkillBuilder-ITSM-Lookup',
        Payload=json.dumps({'ticketNumber': ticketNumber}).encode('utf-8')
    )
    return json.loads(result['Payload'].read())


# Nova Lite is invoked through a cross-Region inference profile. Derive the
# correct geo prefix (us / eu) from the notebook's Region.
geo = 'us' if curr_region.startswith('us-') else 'eu'
bedrock_model = BedrockModel(model_id=f"{geo}.amazon.nova-lite-v1:0")

system_prompt = (
    "You are a helpful assistant that answers questions about the IT Employee handbook "
    "using the retrieve tool to search the knowledge base. You can also create new ITSM "
    "tickets with the create_ticket tool and look up existing tickets with the lookup_ticket "
    "tool. When you create a ticket, return the ticket number to the user. When asked to look "
    "up a ticket, use the lookup_ticket tool and report the status, description, urgency, "
    "impact, and created date. Do not make up any ticket details; they must come from the "
    "lookup_ticket tool."
)

agent = Agent(
    model=bedrock_model,
    system_prompt=system_prompt,
    tools=[retrieve, create_ticket, lookup_ticket]
)

print("Strands agent created.")

Strands agent created.


## Step 11: Test your Strands agent

Now that your agent is built, you test it directly from this notebook. You call the agent like a function. It uses the `retrieve` tool to answer questions from the Knowledge Base, then uses the `create_ticket` and `lookup_ticket` tools to create and look up an ITSM ticket.

In [25]:
# Define a small helper to send a message to the agent. Calling the Strands agent
# runs its reasoning loop and streams the response to the notebook output.

def chat(prompt):
    """Send a message to the agent and print the response."""
    print(f"You: {prompt}\n")
    agent(prompt)
    print("\n" + "-" * 60)

print("Ready to chat with the agent.")

Ready to chat with the agent.


In [26]:
# Test Knowledge Base queries: Asks the agent questions that are answered
# by the Employee IT Handbook stored in the Knowledge Base. The agent uses the
# retrieve tool to find relevant content from the vector store and generates a response.

chat("Tell me the password policy from the IT handbook.")
chat("Tell me the policy on using personal mobile phones from the IT handbook.")
chat("Tell me the remote work policy from the IT handbook.")

You: Tell me the password policy from the IT handbook.

<thinking> The user is asking for the password policy from the IT handbook. I should first try to retrieve this information from the knowledge base. </thinking>

Tool #1: retrieve
<thinking> The password policy has been successfully retrieved from the IT handbook. I will now share this information with the user. </thinking>

Here is the password policy from the IT handbook:
- Passwords must be at least 8 characters long.
- Include a mix of uppercase, lowercase, numbers, and special characters.
- Change passwords every 90 days.
------------------------------------------------------------
You: Tell me the policy on using personal mobile phones from the IT handbook.

<thinking> The user is asking for the policy on using personal mobile phones from the IT handbook. I should first try to retrieve this information from the knowledge base. </thinking> 
Tool #2: retrieve
<thinking> The tool did not find any relevant information on the pol

In [27]:
# Test the create_ticket tool: Sends a complete ticket creation request in a single
# message so the agent has all required parameters upfront. The agent calls the
# create_ticket tool, which invokes the Lambda function to store the ticket in
# DynamoDB, and returns a ticket number.

chat("Please create an INC ticket with the following details: Description: Power outage, Impact on Business: High, Urgency level: Critical")

You: Please create an INC ticket with the following details: Description: Power outage, Impact on Business: High, Urgency level: Critical

<thinking> The user has requested to create an INC ticket with specific details. I will use the create_ticket tool to create this ticket and return the ticket number to the user. </thinking> 
Tool #4: create_ticket
<thinking> The ticket has been successfully created. I will now share the ticket number with the user. </thinking>

Your INC ticket has been created with the following details:
- **Ticket Number:** INC40918362
- **Description:** Power outage
- **Impact on Business:** High
- **Urgency Level:** Critical

Please use this ticket number to follow up on the status of your request.
------------------------------------------------------------


In [28]:
# Test the lookup_ticket tool: Uses the ticket number returned in the previous
# response to look up ticket details via the lookup_ticket tool.
# Replace XXXXXXXXXXXX with the actual ticket number from the response above.

ticket_number = "INC40918362"  # <-- Replace with your ticket number

chat(f"Please look up information for ticket number: {ticket_number}")

You: Please look up information for ticket number: INC40918362

<thinking> The user has asked to look up information for a specific ticket number. I will use the lookup_ticket tool to retrieve the details of this ticket. </thinking> 
Tool #5: lookup_ticket
<thinking> The ticket information has been successfully retrieved. I will now share this information with the user. </thinking>

Here are the details for your ticket (INC40918362):
- **Status:** Pending
- **Description:** Power outage
- **Impact on Business:** High
- **Urgency Level:** Critical
- **Created At:** 2026-10-25 04:37:57 UTC

Please use this information to follow up on the status of your request.
------------------------------------------------------------


### Lab complete!

You have successfully built and tested an autonomous agent with the Strands Agents SDK, backed by an Amazon Bedrock Knowledge Base and custom tools. Return to the lab session to review the conclusion.